# The Kestrel Chase

A courier known as Kestrel removed a reference vial of a vaccine seed strain
from a laboratory in Toronto and left the country. A seed strain is the
starting material from which a manufacturer grows a vaccine; a stolen one can
be sold, copied or destroyed, and each outcome sets a vaccine programme back
by months.

Kestrel made three hops before going to ground. Each hop left a different
kind of evidence, and each kind needs a different technique to read:

| Hop | Evidence | Technique |
|---|---|---|
| 1 | A daylight log: sunrise and sunset times recorded by a phone | Inverting a physical model |
| 2 | Location pings from a second phone, many of them wrong | Robust estimation |
| 3 | Distances to radio relays that picked up a signal | Nonlinear least squares |

Every student in the class holds a different part of each evidence file. All
the parts describe the same route, so no one part is special and no one
student can be skipped. When your three functions pass, your copy of the
route appears on the class map on the projector, added to every other
student's copy. Where most of the class agrees, the route is bright.

## Setup

`chase_helpers.py` provides the solar model, great-circle distances, the
readers for the evidence files and a gazetteer of cities. Each function below
must return a pair `(lat, lon)` in degrees.

The practice files (`practice_*.csv`) come from a different trip, and
`practice_key.json` gives its answers. Test each function on the practice
evidence first, then on your own evidence.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar

from chase_helpers import (
    day_length_hours, equation_of_time, haversine_km, load_cities, load_daylight_log,
    load_pings, load_relays, nearest_city, plot_world, predict_daylight, report,
    solar_noon_minutes,
)

cities = load_cities()
with open("practice_key.json", encoding="utf-8") as handle:
    practice = json.load(handle)

## Hop 1: the daylight log

Kestrel's phone recorded the time of sunrise and sunset each day, in UTC. The
times identify the place, because the Sun's path depends on where the observer
stands.

- **Longitude sets the time of solar noon.** The Sun crosses the meridian
  4 minutes earlier for each degree east. Solar noon is the midpoint of
  sunrise and sunset, and `chase_helpers.solar_noon_minutes` gives the
  relationship: noon in minutes after 00:00 UTC is
  `720 - 4 * lon - equation_of_time(date)`. Solve it for `lon`.
- **Latitude sets the length of the day.** Between May and August, days are
  long in the north and short in the south. `day_length_hours(lat, dates)`
  predicts the day length at a latitude. Find the latitude whose predictions
  best match the observed day lengths, for example by minimizing the sum of
  squared differences with `scipy.optimize.minimize_scalar`.

This is model inversion: a forward model predicts measurements from
parameters, and the inverse problem recovers the parameters from the
measurements. The same reasoning estimates a drug's elimination rate from
blood concentrations measured over time.

In [ ]:
def fit_daylight(log):
    midnight = log["date"]
    rise = (log["sunrise"] - midnight).dt.total_seconds() / 60.0
    sets = (log["sunset"] - midnight).dt.total_seconds() / 60.0

    noon = (rise + sets) / 2.0
    lon = float(np.mean((720.0 - equation_of_time(midnight) - noon) / 4.0))

    observed_hours = (sets - rise) / 60.0

    def squared_error(lat):
        predicted = day_length_hours(lat, midnight)
        if np.isnan(predicted).any():
            return 1e9
        return float(np.sum((predicted - observed_hours.to_numpy()) ** 2))

    lat = minimize_scalar(squared_error, bounds=(-65.0, 65.0), method="bounded").x
    return float(lat), lon

In [ ]:
practice_log = load_daylight_log("practice_daylight_log.csv")
report(fit_daylight, practice_log, cities, practice["hops"][0])

log = load_daylight_log("daylight_log.csv")
log

In [ ]:
report(fit_daylight, log, cities)

## Hop 2: the phone pings

A second phone reported its location every few minutes. Most pings place it
in one city, but three kinds of error contaminate the file:

- Pings at exactly (0, 0), a point in the Gulf of Guinea. A phone that has no
  fix sometimes reports zeros instead of nothing.
- Stale pings from the previous city, replayed from the phone's cache.
- A few pings scattered across the world by a faulty receiver.

Plot the pings on the base map before you write any code. The mean of the
latitudes and the mean of the longitudes is pulled far from the city by these
outliers; the median is not. A robust estimate, such as the median followed
by the mean of the pings near it, finds the city. Wearable sensors in health
studies produce the same kinds of error, and the same methods clean them.

In [ ]:
pings = load_pings("pings.csv")
fig, ax = plt.subplots(figsize=(10, 5))
plot_world(ax)
ax.scatter(pings["lon"], pings["lat"], s=12, color="tab:red")
ax.set_title("Phone pings")
plt.show()

In [ ]:
def locate_from_pings(pings):
    points = pings[(pings["lat"] != 0.0) | (pings["lon"] != 0.0)]
    centre_lat, centre_lon = points["lat"].median(), points["lon"].median()
    near = points[haversine_km(centre_lat, centre_lon, points["lat"], points["lon"]) < 50.0]
    return float(near["lat"].mean()), float(near["lon"].mean())

In [ ]:
report(locate_from_pings, load_pings("practice_pings.csv"), cities, practice["hops"][1])
report(locate_from_pings, pings, cities)

## Hop 3: the relay distances

Four radio relays picked up Kestrel's signal and each recorded how far away
it was, in kilometres, with a few kilometres of error. The point you want is
the one whose great-circle distances to the relays best match the recorded
distances.

There is no closed form for that point on a sphere, so solve for it
numerically. Write a function of a candidate `(lat, lon)` that returns the
four differences between the computed distances (`haversine_km`) and the
recorded ones, and pass it to `scipy.optimize.least_squares` with a starting
guess. The centre of the relays is a reasonable start, but it is not the
answer. Ambulance dispatch locates a caller's phone from its distances to cell
towers by the same method.

In [ ]:
def trilaterate(relays):
    lats, lons = relays["lat"].to_numpy(), relays["lon"].to_numpy()
    measured = relays["distance_km"].to_numpy()

    def residuals(point):
        return haversine_km(point[0], point[1], lats, lons) - measured

    start = [float(np.mean(lats)), float(np.mean(lons))]
    fit = least_squares(residuals, start, bounds=([-90.0, -180.0], [90.0, 180.0]))
    return float(fit.x[0]), float(fit.x[1])

In [ ]:
report(trilaterate, load_relays("practice_relays.csv"), cities, practice["hops"][2])
report(trilaterate, load_relays("relays.csv"), cities)

## After the chase

Submit the notebook. The tests run your three functions on your part of the
evidence, and a fifth test runs them on evidence from a route that is not
in this notebook, so a function must compute its answer rather than return
a known city.

Two questions for discussion:

1. The route on the class map is curved, although each hop is the shortest
   path between two cities. Explain why, with reference to the map's
   projection.
2. Which hop's estimate has the largest error, in kilometres? Identify the
   source of that error in the evidence.